In [ ]:
# 파일 저장을 위한 구글 드라이브 마운트

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# ◆ 0. 샘플 데이터 불러오기

In [ ]:
import pandas as pd

# ◆ 1. 데이터 전처리하기
* 1.1. 특수문자 및 숫자 제거
* 1.2. 의미없는 짧은 글 제거

### 1.1. 특수문자 및 숫자 제거
* 정규표현식을 사용해봅시다 😸

In [ ]:
# 라이브러리/모듈 import 하기

import re # 정규표현식을 사용하기 위한 파이썬 기본 내장 라이브러리
from tqdm import tqdm # 반복문이 얼마나 진행됐는지 진행률 바로 보여주는 라이브러리

### 1.2. 의미없는 짧은 글 제거
* 15자 미만인 리뷰를 제거해봐요 😸

In [ ]:
# 리뷰를 한 줄씩 확인하면서 15자 미만이면 해당 행을 직접 삭제 -- enumerate 활용


`enumerate()` : 반복 가능한 객체를 입력 받아, 각 요소에 대해 인덱스와 값을 함께 반환하는 파이썬 내장 함수입니다.


```
    fruits = ['apple', 'banana', 'cherry']
    for i, fruit in enumerate(fruits):
        print(i, fruit)
```
    
  즉, `enumerate(fruits)`는 내부적으로 다음과 같은 튜플을 생성합니다 :
    
  ```
  [(0, 'apple'), (1, 'banana'), (2, 'cherry')]
  ```
    
  ➡️ 반복문을 돌면서, 지금 몇 번 째 반복인지’를 자동으로 알려줄 수 있습니다 (i 값)

In [ ]:
# 행 삭제 후 뒤죽박죽이 된 인덱스를 0부터 다시 정렬하기


# ◆ 2. 데이터 형태소 분리하기
(제공되는 한국어 불용어 파일 사용: ko-stopwords.csv)
* 2.1 불용어적용 및 형태소 분리
* 2.2 데이터 프레임에 추가

In [ ]:
pip install konlpy

In [ ]:
# Okt(Open Korean Text)는 트위터가 만든 한국어 형태소 분석기로,
# 한국어 문장을 단어 단위로 쪼개고 각 단어의 품사(명사/동사/형용사 등)를 태깅해주는 도구입니다.

from konlpy.tag import Okt
okt = Okt()                 # 형태소 분석기 객체를 생성

### 2.1. 불용어적용 및 형태소 분리

In [ ]:
# 불용어 파일 가져오기


### 2.2 데이터 프레임에 추가

# ◆ 3. 벡터화
* 3.1 doc2vec 준비(문서의 순서 매기기)
* 3.2 doc2vec 학습시키기
* 3.3 벡터 값 데이터 프레임에 추가

In [ ]:
pip install gensim

In [ ]:
import gensim # 자연어 처리 및 토픽 모델링을 위한 라이브러리
from gensim.models.doc2vec import TaggedDocument # gensim의 doc2vec 모듈에서 문서에 태그(레이블)를 함께 저장하기 위한 TaggedDocument 클래스
from gensim.models import Doc2Vec # 문서 임베딩 학습을 위한 Doc2Vec 모델 클래스

### 3.1 doc2vec 준비 : 각 문서에 고유 ID(태그)를 붙인 TaggedDocument 객체 리스트 생성
* word2vec은 단어 하나를 하나의 vector화 (단어 1개 → 벡터 1개)
* doc2vec은 문서 하나를 하나의 vector화 (문서 1개 → 벡터 1개; 문서 전체의 의미를 하나의 숫자 배열로 표현)

TaggedDocument란? Doc2Vec에게 학습 데이터를 넘겨줄 때 사용하는 전용 그릇(객체)입니다.

```
TaggedDocument(
    words = ["냉장고", "소음", "크다"],  # 단어 리스트
    tags  = ["document0"]               # 이 문서의 고유 ID
)
```

words: 분석할 단어 목록["냉장고", "소음", "크다"]
tags: 이 문서를 구별하는 ID["document0"]

왜 일반 리스트가 아니라 TaggedDocument를 쓸까요?


```
# 일반 리스트만 있는경우
["냉장고", "소음", "크다"]   # "이게 몇 번 문서야?" → Doc2Vec이 모름

# TaggedDocument를 쓰면
TaggedDocument(
    words=["냉장고", "소음", "크다"],
    tags=["document0"]               # "0번 문서야!" → Doc2Vec이 기억
)
```

Doc2Vec은 단어의 의미뿐 아니라 "이 단어들이 어느 문서에 속하는가" 도 함께 학습합니다.
그래서 반드시 문서 ID(tags)가 붙어 있어야 합니다.

### 3.2 doc2vec 학습시키기

####  하이퍼파라메터 설명
**1. vector_size → 문서 임베딩 차원 수**
* 소량 데이터(문서 수 몇 천 단위): 100 ~ 200(권장)
* 중간 이상 데이터(문서 수 수만 이상): 200 ~ 400(권장)
* 너무 크게 잡으면 학습 시간이 증가하고, 데이터가 적으면 과적합 위험이 있습니다.
* 처음에는 200 또는 300 정도에서 시작 후, 성능 보고 조정하는 패턴을 추천합니다.

**2. alpha, min_alpha → 초기 학습률 / 마지막 학습률**
*  alpha: 흔히 0.025 ~ 0.05 정도에서 시작하며, 값이 클수록 빠르게 학습하지만 불안정해질 수 있습니다.
* min_alpha: 학습 후반에 내려갈 학습률로, 보통 0.0001 ~ 0.001 정도로 두는 경우가 많습니다.
* 실무에서 자주 쓰는 조합 예시: alpha=0.025, min_alpha=0.0001
* 너무 큰 alpha → 학습이 요동
* min_alpha를 너무 크게 두면 → 마지막까지 파라미터가 너무 많이 움직임

**3. window → 주변 단어를 몇 개까지 컨텍스트로 볼지**
* 리뷰/짧은 문장 위주 텍스트: 3 ~ 5(권장)
* 긴 문서, 문맥 넓게 보고 싶을 때: 5 ~ 10(권장)
* 너무 작으면 문맥 정보 부족, 너무 크면 관계 없는 단어까지 섞여 노이즈 증가합니다.
* 지금 설정한 window=3은 짧은 리뷰 기준으로는 무난한 값으로, 좀 더 넓게 의미를 보고 싶다면 5 정도도 많이 사용합니다.

**4. min_count → 최소 몇 번 이상 등장한 단어만 학습에 포함할지**
* 일반적인 텍스트 마이닝: 2 ~ 5(권장)
* 데이터가 매우 적은 경우 불가피하게 1을 쓰기도 합니다. 다만 노이즈가 많아질 수 있어요! (오타, 고유명사, 희귀 단어까지 모두 포함하기 때문에)
* 리뷰 데이터가 수천 개 이상이라면: min_count=2 또는 3 정도로 시작하는 경우가 많습니다.

**5. dm → 학습 방식 선택 (Doc2Vec 알고리즘 타입)**
* dm=1 : DM(Distributed Memory) 방식으로 문맥 + 문서 벡터를 함께 사용합니다.
  * 문서 의미를 안정적으로 잡는 데 자주 사용합니다.
* dm=0 : DBOW(Distributed Bag of Words) 방식으로, word2vec의 Skip-gram과 비슷합니다.
  * 종종 더 빠르고, word 벡터 성능이 좋은 경우 있습니다.
  * 고급 튜닝에서는 dm=1 모델, dm=0 모델 둘 다 학습한 뒤, 벡터를 이어 붙이거나(concatenate) 둘 중 더 좋은 쪽을 선택하기도 합니다.

### 3.3 벡터 값 데이터 프레임에 추가

# ◆ 4. 병합 계층적 클러스터링
* 4.1 ward 기준으로 덴드로그램 그려보기
* 4.2 실루엣 지수 확인해서 토픽 갯수 정하기
* 4.3 가장 적절한 클러스터링 갯수 df에 삽입

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage  # 계층적 군집 분석(linkage)과 시각화(dendrogram) 함수 (그림을 그려줄 수 있는 친구)
from matplotlib import pyplot as plt                     # 그래프 시각화 라이브러리

### 4.1 ward 기준으로 덴드로그램 그려보기
- 덴드로그램 : 문서들이 어떻게 묶이는지를 나무(tree) 형태로 보여주는 그래프입니다.
- 가지가 합쳐지는 높이(거리)가 클수록 두 군집이 서로 다르다는 뜻입니다.

### 4.2 실루엣 지수 확인해서 토픽 갯수 정하기
- 실루엣 지수: 군집화가 얼마나 잘 됐는지 평가하는 점수 (-1 ~ 1) 입니다.
- 1에 가까울수록 같은 군집끼리는 가깝고, 다른 군집과는 멀리 떨어져 있음을 의미합니다.

In [ ]:
from sklearn.metrics.cluster import silhouette_score    # 실루엣 점수 계산 함수
from sklearn.cluster import AgglomerativeClustering     # 병합형 계층 군집 알고리즘 (클러스터 분리를 할 때 사용함)

### 4.3 가장 적절한 클러스터링 갯수 df에 삽입

# ◆ 5. 해석하기:TF-IDF
* 문서 내에서 어떤 단어가 얼마나 중요한지를 평가하는 데 사용되는 방법
→ 각 문서에서의 핵짐 주제열을 판단할 수 있는 빈도분석의 기법
* 5.1 TF-IDF 계산
* 5.2 데이터프레임으로 만들고 정렬하기

### 5.1 TF-idf 계산
* 각 클러스터 마다 tfidf가 높은 워드들 찾기
* 각 클러스터들을 하나의 문서로 가정하여 tf-idf 값 추출

In [ ]:
from collections import Counter                              # 각 클러스터 내 단어 빈도 등을 계산할 때, 원소의 출현 횟수를 손쉽게 세기 위한 Counter 클래스
import numpy as np                                           # 수치 연산, 배열 처리, 벡터/행렬 연산 등을 위해 numpy 라이브러리
from sklearn.feature_extraction.text import TfidfVectorizer # 텍스트 데이터를 TF-IDF(단어 빈도-역문서 빈도) 방식의 수치 벡터로 변환하기 위한 TfidfVectorizer 클래스

### 5.2 데이터프레임으로 만들고 정렬하기